# PPO en Atari (Colab): agente antes y después de entrenar

En este notebook vas a:

1. instalar las librerías necesarias,
2. crear un agente **PPO** con **Stable-Baselines3**,
3. ver una **animación antes de entrenar**,
4. entrenarlo en un juego de Atari,
5. ver una **animación después de entrenar**,
6. guardar el modelo entrenado.

He elegido **Pong** porque suele ser uno de los Atari más razonables para una demo con PPO.

> **Nota importante sobre tiempo de entrenamiento**  
> En RL, una demo rápida no siempre produce un agente "bueno".  
> Por eso el notebook trae un valor por defecto moderado y te deja una recomendación clara para aumentar los timesteps si quieres una mejora más visible.

## 1) Instalación

Esta celda está pensada para **Google Colab**.

In [ ]:
# Instalación para Colab
!pip -q install "stable-baselines3[extra]" "gymnasium[atari,accept-rom-license]" imageio imageio-ffmpeg

## 2) Imports y configuración general

In [ ]:
import os
import random
import warnings

import numpy as np
import torch
import gymnasium as gym
import ale_py

# En versiones recientes de Gymnasium/ALE conviene registrar los entornos explícitamente
try:
    gym.register_envs(ale_py)
except Exception:
    pass

import imageio.v2 as imageio
from IPython.display import Image, display

from stable_baselines3 import PPO
from stable_baselines3.common.atari_wrappers import AtariWrapper
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.vec_env import VecFrameStack

warnings.filterwarnings("ignore")

# ---------------------------
# Configuración principal
# ---------------------------
SEED = 42
ENV_ID = "PongNoFrameskip-v4"

# Entornos paralelos para entrenamiento
N_ENVS = 4

# Demo rápida: útil para probar que todo funciona.
# Si quieres una mejora más clara, sube esto a 300_000, 500_000 o 1_000_000.
TRAINING_TIMESTEPS = 100_000

# Duración máxima de cada animación
MAX_GIF_STEPS = 2_000
GIF_FPS = 20
GIF_EVERY_N_FRAMES = 2   # para que el GIF pese menos

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo detectado:", device)
print("Juego Atari:", ENV_ID)
print("Timesteps de entrenamiento:", TRAINING_TIMESTEPS)

## 3) Funciones auxiliares

Vamos a crear:

- un entorno vectorizado para **entrenar**,
- un entorno de un solo episodio para **grabar animaciones**,
- una función de **evaluación rápida**.

In [ ]:
def make_train_env(n_envs=N_ENVS, seed=SEED):
    """
    Entorno para entrenar PPO.
    Usamos el preprocesado clásico de Atari:
    - frame skip
    - resize a 84x84
    - escala de grises
    - clipping de recompensas
    - frame stack de 4 imágenes
    """
    env = make_vec_env(
        ENV_ID,
        n_envs=n_envs,
        seed=seed,
        wrapper_class=AtariWrapper,
        wrapper_kwargs=dict(
            terminal_on_life_loss=False,
            clip_reward=True,
            frame_skip=4,
            screen_size=84,
        ),
    )
    env = VecFrameStack(env, n_stack=4)
    return env


def make_record_env(seed=SEED):
    """
    Entorno para grabar un episodio y convertirlo en GIF.
    Aquí desactivamos el clipping de recompensa para que la evaluación
    sea un poco más interpretable, aunque el modelo se haya entrenado con clipping.
    """
    env = make_vec_env(
        ENV_ID,
        n_envs=1,
        seed=seed,
        env_kwargs=dict(render_mode="rgb_array"),
        wrapper_class=AtariWrapper,
        wrapper_kwargs=dict(
            terminal_on_life_loss=False,
            clip_reward=False,
            frame_skip=4,
            screen_size=84,
        ),
    )
    env = VecFrameStack(env, n_stack=4)
    return env


def save_policy_gif(
    model,
    out_path,
    deterministic=True,
    max_steps=MAX_GIF_STEPS,
    fps=GIF_FPS,
    every_n_frames=GIF_EVERY_N_FRAMES,
):
    """
    Ejecuta un episodio con el modelo y guarda un GIF.
    """
    folder = os.path.dirname(out_path)
    if folder:
        os.makedirs(folder, exist_ok=True)

    env = make_record_env()
    obs = env.reset()

    frames = []
    total_reward = 0.0
    done = False
    step = 0

    while (not done) and (step < max_steps):
        frame = env.render()
        if frame is not None:
            frames.append(frame)

        action, _ = model.predict(obs, deterministic=deterministic)
        obs, rewards, dones, infos = env.step(action)

        total_reward += float(rewards[0])
        done = bool(dones[0])
        step += 1

    env.close()

    if len(frames) == 0:
        raise RuntimeError(
            "No se capturaron frames. Revisa la instalación de Atari o el render_mode."
        )

    frames = frames[::every_n_frames]
    imageio.mimsave(out_path, frames, fps=fps)

    return out_path, total_reward, len(frames)


def show_gif(path):
    display(Image(filename=path))


def quick_eval(model, n_eval_episodes=5):
    """
    Evaluación rápida del modelo.
    """
    env = make_record_env()
    mean_reward, std_reward = evaluate_policy(
        model,
        env,
        n_eval_episodes=n_eval_episodes,
        deterministic=True,
    )
    env.close()
    return mean_reward, std_reward

## 4) Crear el agente PPO (todavía sin entrenar)

Usamos una **CNN policy** porque las observaciones son imágenes.

In [ ]:
train_env = make_train_env()

model = PPO(
    policy="CnnPolicy",
    env=train_env,
    learning_rate=2.5e-4,
    n_steps=128,
    batch_size=256,
    n_epochs=4,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.1,
    ent_coef=0.01,
    vf_coef=0.5,
    verbose=1,
    tensorboard_log="./ppo_atari_tensorboard/",
    device=device,
    seed=SEED,
)

print(model.policy)

## 5) Animación **antes** de entrenar

Aquí el agente todavía no sabe jugar.  
La política existe, pero sus pesos están sin aprender.

In [ ]:
before_gif, before_reward, before_nframes = save_policy_gif(
    model,
    out_path="gifs/pong_before_training.gif",
    deterministic=False,   # antes de entrenar tiene sentido verlo más "aleatorio"
)

print(f"GIF guardado en: {before_gif}")
print(f"Recompensa del episodio (sin entrenar): {before_reward:.2f}")
print(f"Número de frames guardados: {before_nframes}")

show_gif(before_gif)

## 6) Entrenamiento

> Si estás en Colab con GPU, normalmente irá mejor.  
> Si el aprendizaje te parece escaso, sube `TRAINING_TIMESTEPS`.

También puedes abrir TensorBoard después si quieres inspeccionar el proceso con más detalle.

In [ ]:
model.learn(
    total_timesteps=TRAINING_TIMESTEPS,
    progress_bar=True,
)

model.save("ppo_pong_colab")
print("Modelo guardado como: ppo_pong_colab.zip")

## 7) Animación **después** de entrenar

Ahora volvemos a ejecutar un episodio con la política ya ajustada.

In [ ]:
after_gif, after_reward, after_nframes = save_policy_gif(
    model,
    out_path="gifs/pong_after_training.gif",
    deterministic=True,
)

print(f"GIF guardado en: {after_gif}")
print(f"Recompensa del episodio (después de entrenar): {after_reward:.2f}")
print(f"Número de frames guardados: {after_nframes}")

show_gif(after_gif)

## 8) Evaluación rápida

Esto te da una idea de si el agente está mejorando.

In [ ]:
mean_reward, std_reward = quick_eval(model, n_eval_episodes=5)
print(f"Recompensa media en 5 episodios: {mean_reward:.2f} ± {std_reward:.2f}")

## 9) (Opcional) TensorBoard en Colab

Si quieres ver curvas de entrenamiento:

In [ ]:
# Descomenta estas líneas si quieres usar TensorBoard en Colab
# %load_ext tensorboard
# %tensorboard --logdir ./ppo_atari_tensorboard/

## 10) Consejos prácticos

### Si el agente mejora poco
Prueba una de estas opciones:

- subir `TRAINING_TIMESTEPS` a `300_000`, `500_000` o `1_000_000`,
- dejar `N_ENVS = 4` o subirlo si tu runtime aguanta,
- repetir el experimento con otra seed.

### Si el GIF pesa demasiado
Puedes:

- subir `GIF_EVERY_N_FRAMES` a `3` o `4`,
- bajar `MAX_GIF_STEPS`,
- bajar `GIF_FPS`.

### Si falla la creación del entorno Atari
Haz dos cosas:

1. vuelve a ejecutar la celda de instalación,
2. reinicia el runtime de Colab y ejecuta el notebook desde arriba.

### Otros juegos que puedes probar
Solo cambia:

```python
ENV_ID = "BreakoutNoFrameskip-v4"
```

o por ejemplo:

```python
ENV_ID = "SpaceInvadersNoFrameskip-v4"
```

y vuelve a ejecutar todo el notebook.